# Pachete necesare pentru folosirea acestui Notebook

Vom folosi [scipy](https://scipy.org/), [numpy](https://numpy.org/) și [matplotlib](https://matplotlib.org/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import datasets, ndimage # misc is deprecated
from scipy.fft import dctn, idctn
import imageio.v3 as io

# Imaginea cu care lucrăm

Vom folosi o imagine din setul de date oferit implicit de către scipy.

In [ ]:
X = datasets.ascent()
plt.imshow(X, cmap=plt.cm.gray)
plt.show()

# Transformata DCT a unei imagini

Transformata DCT se extinde la mai multe dimensiuni similar cu transformata DFT. Pentru un semnal bidimensional, precum o imagine, DCT-II devine:

$$
Y_{m_1,m_2} = \sum_{n_1=0}^{N_1-1}
              \sum_{n_2=0}^{N_2-1}
              x_{n_1,n_2}
              \cos\left[\frac{\pi}{N_1}m_1
                \left(n_1 + \frac12\right)\right]
              \cos\left[\frac{\pi}{N_2}m_2\left(n_2 + \frac12\right)\right]
$$

* unde $n_1$ și $n_2$ sunt pozițile pixelilor pe orizontală, respectiv, pe verticală
* bin-urile rezultate corespund pozițiilor pixelilor
* spectrul este în continuare simetric și par
* proprietățile transformatei DCT-II sunt respectate și în cazul celei 2D

În Python avem rutina `scipy.fft.dct` pentru 1D și `scipy.fft.dctn` pentru generalizarea la semnale $n$-dimensionale. Dimensiunea este determinată automat după forma semnalului; tipul DCT poate fi specificat prin atributul `type` (implicit II).

In [ ]:
Y1 = dctn(X, type=1)
Y2 = dctn(X, type=2)
Y3 = dctn(X, type=3)
Y4 = dctn(X, type=4)
freq_db_1 = 20*np.log10(abs(Y1))
freq_db_2 = 20*np.log10(abs(Y2))
freq_db_3 = 20*np.log10(abs(Y3))
freq_db_4 = 20*np.log10(abs(Y4))

plt.subplot(221).imshow(freq_db_1)
plt.subplot(222).imshow(freq_db_2)
plt.subplot(223).imshow(freq_db_3)
plt.subplot(224).imshow(freq_db_4)
plt.show()

# Compactarea energiei. Compresie.

Putem profita de proprietatea compresiei energiei prin anularea frecvențelor DCT începând cu *bin*-ul `k` după care aplicăm transformata DCT inversă (similar cu tema anterioară):

In [ ]:
k = 120

Y_ziped = Y2.copy()
Y_ziped[k:] = 0
X_ziped = idctn(Y_ziped)

plt.imshow(X_ziped, cmap=plt.cm.gray)
plt.show()

# JPEG

Algoritmul de compresie JPEG are patru etape:

1. transformarea imaginii din pixeli RGB în Y'CbCr
2. aplicarea 2D-DCT pe blocuri disincte de 8x8 pixeli din imagine
3. cuantizarea în frecvență cu $Q$ dat de standardul JPEG
4. compresia rezultatului cu coduri Huffman

Unde matricea JPEG de cuantizare $Q$ este:
$$
Q =
\begin{bmatrix}
16 & 11 & 10 & 16 & 24 & 40 & 51 & 61 & \\
12 & 12 & 14 & 19 & 26 & 28 & 60 & 55 & \\
14 & 13 & 16 & 24 & 40 & 57 & 69 & 56 & \\
14 & 17 & 22 & 29 & 51 & 87 & 80 & 62 & \\
18 & 22 & 37 & 56 & 68 & 109 & 103 & 77 & \\
24 & 35 & 55 & 64 & 81 & 104 & 113 & 92 & \\
49 & 64 & 78 & 87 & 103 & 121 & 120 & 101\\
72 & 92 & 95 & 98 & 112 & 100 & 103 & 99\\
\end{bmatrix}
$$

Imaginea noastră de test este monocromă, deci nu necesită pasul 1, dar putem efectua o operație de *down-sampling* în preprocesare precum am prezentat la curs.

In [ ]:
Q_down = 10

X_jpeg = X.copy()
X_jpeg = Q_down*np.round(X_jpeg/Q_down);

plt.subplot(121).imshow(X, cmap=plt.cm.gray)
plt.title('Original')
plt.subplot(122).imshow(X_jpeg, cmap=plt.cm.gray)
plt.title('Down-sampled')
plt.show()

Pentru fiecare bloc de $8\times 8$ aplică DCT și cuantizare.

In [ ]:
Q_jpeg = [[16, 11, 10, 16, 24, 40, 51, 61],
          [12, 12, 14, 19, 26, 28, 60, 55],
          [14, 13, 16, 24, 40, 57, 69, 56],
          [14, 17, 22, 29, 51, 87, 80, 62],
          [18, 22, 37, 56, 68, 109, 103, 77],
          [24, 35, 55, 64, 81, 104, 113, 92],
          [49, 64, 78, 87, 103, 121, 120, 101],
          [72, 92, 95, 98, 112, 100, 103, 99]]

# Encoding
x = X[:8, :8]
y = dctn(x)
y_jpeg = Q_jpeg*np.round(y/Q_jpeg)

# Decoding
x_jpeg = idctn(y_jpeg)

# Results
y_nnz = np.count_nonzero(y)
y_jpeg_nnz = np.count_nonzero(y_jpeg)

plt.subplot(121).imshow(x, cmap=plt.cm.gray)
plt.title('Original')
plt.subplot(122).imshow(x_jpeg, cmap=plt.cm.gray)
plt.title('JPEG')
plt.show()

print('Componente în frecvență:' + str(y_nnz) + 
      '\nComponente în frecvență după cuantizare: ' + str(y_jpeg_nnz))

# Sarcini

1. [6p] Completați algoritmul JPEG incluzând toate blocurile din imagine.

2. [4p] Extindeți la imagini color (incluzând transformarea din RGB în Y'CbCr).

3. [6p] Extindeți algoritmul pentru compresia imaginii până la un prag MSE impus de utilizator.

4. [4p] Extindeți algoritmul pentru compresie video. Demonstrați pe un clip scurt din care luați fiecare cadru și îl tratați ca pe o imagine.

# Sarcina 1

In [ ]:
def jpeg_gray_compression(X, qf=1, plot=False):
    h, w = X.shape
    h_new = (h // 8) * 8
    w_new = (w // 8) * 8
    
    # The output image
    X_jpeg = np.zeros((h_new, w_new))
    
    Q = np.array(Q_jpeg)
    q = Q * qf

    y_nnz = 0
    y_jpeg_nnz = 0


    for i in range(0, h_new, 8):
        for j in range(0, w_new, 8):
            row_start, row_end = i, i+8
            col_start, col_end = j, j+8
            
            x = X[row_start:row_end, col_start:col_end]

            y = dctn(x)
            y_nnz += np.count_nonzero(y)

            y_jpeg = q * np.round(y / q)
            y_jpeg_nnz += np.count_nonzero(y_jpeg)

            X_jpeg[row_start:row_end, col_start:col_end] = idctn(y_jpeg)

    if plot:
        plt.subplot(121).imshow(X, cmap=plt.cm.gray)
        plt.title('Original')
        plt.subplot(122).imshow(X_jpeg, cmap=plt.cm.gray)
        plt.title(f'JPEG (qf = {qf})')
        plt.savefig('jpeg_gray_compression.pdf')
        plt.show()

        print('Componente în frecvență: ' + str(y_nnz) + 
            '\nComponente în frecvență după cuantizare: ' + str(y_jpeg_nnz))
        
        print(f'Rata de compresie: {y_nnz / y_jpeg_nnz:.2f}')

    return X_jpeg


compressed_image = jpeg_gray_compression(X, qf=100, plot=True)

# Sarcina 2

In [ ]:
def rgb2ycbcr(v):
    R = v[:,:,0]
    G = v[:,:,1]
    B = v[:,:,2]
    
    Y  = 0.299 * R + 0.587 * G + 0.114 * B
    Cb = 128.0 + (-0.168736 * R -  0.331264 * G + 0.5 * B) 
    Cr = 128.0 + (0.5 * R -  0.418688 * G -  0.081312 * B) 
    
    return np.stack((Y, Cb, Cr), axis=2)

In [ ]:
def ycbcr2rgb(v):
    Y = v[:,:,0]
    Cb = v[:,:,1] - 128
    Cr = v[:,:,2] - 128
    
    R = Y + 1.402 * Cr
    G = Y - 0.344136 * Cb - 0.714136 * Cr
    B = Y + 1.772 * Cb
    
    R = np.clip(R, 0, 255)
    G = np.clip(G, 0, 255)
    B = np.clip(B, 0, 255)
    
    return np.stack((R, G, B), axis=2).astype(np.uint8)
    

In [ ]:
def jpeg_color_compression(img_rgb, qf=1, plot=False):
    y = rgb2ycbcr(img_rgb)
    
    y[:,:,0] = jpeg_gray_compression(y[:,:,0], qf, False)
    y[:,:,1] = jpeg_gray_compression(y[:,:,1], qf, False)
    y[:,:,2] = jpeg_gray_compression(y[:,:,2], qf, False)
    
    x = ycbcr2rgb(y)

    if plot:
        plt.figure(figsize=(20, 10))
        plt.subplot(121).imshow(img_rgb, cmap=plt.cm.gray)
        plt.title('Original')
        plt.subplot(122).imshow(x, cmap=plt.cm.gray)
        plt.title(f'JPEG (qf = {qf})')
        plt.savefig('jpeg_color_compression.pdf')
        plt.show()
        
    return x

In [ ]:
raccoon_image = datasets.face()
result_image = jpeg_color_compression(raccoon_image, qf=1000, plot=True)

# Sarcina 3

In [ ]:
def mse_jpeg_adaptive(X, threshold_mse, rgb=False, plot=False, figname='jpeg_mse_compression.pdf'):
    # Cropping the original image so its dimensions are multiples of 8
    h, w = X.shape[:2]
    h_new = (h // 8) * 8
    w_new = (w // 8) * 8
    X = X[:h_new, :w_new]

    qf = 10.0
    mse_current = np.inf
    y = None

    while mse_current > threshold_mse and qf > 0.1:

        if rgb:
            y = jpeg_color_compression(X, qf=qf, plot=False)
        else:
            y = jpeg_gray_compression(X, qf=qf, plot=False)

        mse_current = np.mean((y.astype(float) - X.astype(float)) ** 2)
        
        print(f"\tQF: {qf:.2f} -> MSE: {mse_current:.2f}")
        if mse_current > threshold_mse:
            qf *= 0.99
            
    print(f"FINAL -> QF găsit: {qf:.3f} | MSE final: {mse_current:.2f}")

    if plot:
        plt.figure(figsize=(12, 6))
        
        plt.subplot(121)
        if rgb:
            plt.imshow(X.astype(np.uint8))
        else:
            plt.imshow(X, cmap='gray')
        plt.title('Original (Cropped)')
        plt.axis('off')

        plt.subplot(122)
        if rgb:
            plt.imshow(y.astype(np.uint8))
        else:
            plt.imshow(y, cmap='gray')
        plt.title(f'JPEG Optimizat\nMSE: {mse_current:.1f} (Th={threshold_mse}), QF={qf:.2f}')
        plt.axis('off')
        plt.savefig(figname)

        plt.show()
    
    return y

In [ ]:
X_gray = datasets.ascent()
res_gray = mse_jpeg_adaptive(X_gray, threshold_mse=10, rgb=False, plot=True, figname='jpeg_mse_compression_gray.pdf')

In [ ]:
X_color = datasets.face()
res_color = mse_jpeg_adaptive(X_color, threshold_mse=3, rgb=True, plot=True, figname='jpeg_mse_compression_color.pdf')

# Sarcina 4

**Notă privind orientarea video**: Deși videoclipurile filmate cu telefonul apar verticale la redare, ele sunt adesea stocate tehnic pe orizontală (landscape) la nivel de senzori. Orientarea corectă este dictată de un flag de rotație din metadatele fișierului. Deoarece algoritmul nostru procesează direct fluxul de pixeli (raw frames) și ignoră containerul original de metadate, videoclipul rezultat pierde instrucțiunea de rotație și apare orizontal. Pentru a corecta acest lucru, este necesară rotirea manuală a matricei de pixeli.

In [ ]:
def video_jpg_mov(input_path, output_path='video_jpeg_rezultat.mov', qf=10, rotate=False):
    
    try:
        metadata = io.immeta(input_path, plugin="pyav")
        fps = metadata.get('fps', 30)
    except:
        fps = 30
    
    with io.imopen(output_path, 'w', plugin='pyav') as writer:
        writer.init_video_stream('libx264', fps=fps)
    
        frame_count = 0
        
        for frame in io.imiter(input_path, plugin="pyav"):
            frame_count += 1
            if frame_count % 10 == 0:
                print(f"Frame {frame_count} is processing...")

            if rotate:
                frame = np.rot90(frame, k=-1) 
                frame = np.ascontiguousarray(frame)

            compressed_frame = jpeg_color_compression(frame, qf=qf, plot=False)
            compressed_frame = compressed_frame.astype(np.uint8)

            writer.write_frame(compressed_frame)

    print(f"Succes! Saved at: {output_path}")

# Input: .mov | Output: .mov
qf = 500
video_jpg_mov('video_original.mov', f'video_jpeg_qf{qf}.mov', qf=qf, rotate=True)

The Original Video has 12.1 MB

qf=500 - Compressed Video has 3.4 MB

qf=1000 - Compressed Video has 1.6 MB